In [1]:
import cv2
import numpy as np

# Assuming the KalmanFilter3D class is defined as before
class KalmanFilter3D:
    def __init__(self, dt=0.1):
        self.dt = dt
        self.F = np.array([
            [1, 0, 0, dt, 0, 0, 0.5*dt**2, 0, 0],
            [0, 1, 0, 0, dt, 0, 0, 0.5*dt**2, 0],
            [0, 0, 1, 0, 0, dt, 0, 0, 0.5*dt**2],
            [0, 0, 0, 1, 0, 0, dt, 0, 0],
            [0, 0, 0, 0, 1, 0, 0, dt, 0],
            [0, 0, 0, 0, 0, 1, 0, 0, dt],
            [0, 0, 0, 0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 1]
        ])
        self.H = np.array([
            [1, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0, 0, 0, 0]
        ])
        self.Q = np.eye(9) * 0.1
        self.R = np.eye(3) * 0.1
        self.x = {}
        self.P = {}
        self.I = np.eye(9)

    def predict(self, id):
        self.x[id] = self.F @ self.x[id]
        self.P[id] = self.F @ self.P[id] @ self.F.T + self.Q

    def update(self, id, x, y, z=0):
        if id not in self.x:
            self.x[id] = np.zeros((9, 1))
            self.P[id] = np.eye(9)

        z = np.array([[x], [y], [z]])

        self.predict(id)

        K = self.P[id] @ self.H.T @ np.linalg.inv(self.H @ self.P[id] @ self.H.T + self.R)
        self.x[id] = self.x[id] + K @ (z - self.H @ self.x[id])
        self.P[id] = (self.I - K @ self.H) @ self.P[id]

        return {
            "state_estimate": self.x[id],
            "covariance_estimate": self.P[id]
        }


In [ ]:
# Initialize video capture
cap = cv2.VideoCapture('rolling_ball.mp4')

# Initialize Kalman filter
kf = KalmanFilter3D()

# Initialize a dictionary to store Kalman filters for each ball
kalman_filters = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Detect balls in the frame (using a detection algorithm)
    detections = detect_balls(frame)  # This function should return a list of bounding boxes

    detected_ids = set()

    for detection in detections:
        ball_id = 1 #detection['id']
        x, y = detection[0], detection[1]
        detected_ids.add(ball_id)

        if ball_id not in kalman_filters:
            kalman_filters[ball_id] = KalmanFilter3D()

        result = kalman_filters[ball_id].update(ball_id, x, y, 0)

        # Draw the predicted position on the frame
        predicted_position = result["state_estimate"][:2].flatten()
        cv2.circle(frame, (int(predicted_position[0]), int(predicted_position[1])), 5, (0, 255, 0), -1)

    # Predict positions for undetected balls
    for ball_id in kalman_filters.keys():
        if ball_id not in detected_ids:
            kalman_filters[ball_id].predict(ball_id)
            predicted_position = kalman_filters[ball_id].x[ball_id][:2].flatten()
            cv2.circle(frame, (int(predicted_position[0]), int(predicted_position[1])), 5, (0, 0, 255), -1)

    # Display the frame
    cv2.imshow('Frame', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()